In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("IMDB Dataset.csv")

In [3]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
data.drop_duplicates(inplace = True)


In [5]:
data.shape

(49582, 2)

Data Preprocessing

1. Converting to lower case

In [6]:
data["review"] = data["review"].str.lower()

2. Removeing the URLs

In [7]:
import re

def remove_urls(text):
    text = re.sub(r"https\S+", "", text) #patter, replace with, string
    return text

data["review"] = data["review"].apply(remove_urls)

3. Removeing the Punctuations

In [8]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s+]","",text)
    return text
data["review"] = data["review"].apply(remove_punctuations)

4. Remove HTML

In [9]:
def remove_HTML(text):
    text = re.sub(r"<.*?>", "", text)
    return text
data["review"] = data["review"].apply(remove_HTML)

5. Removeing the Stopwords

In [10]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to C:\Users\Krishna
[nltk_data]     Verma\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Krishna
[nltk_data]     Verma\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Krishna
[nltk_data]     Verma\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [11]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [12]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word,"")
    return text


data["review"] = data["review"].apply(remove_stopwords)

In [13]:
data.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


6. Stemming

In [14]:
# running -> run
# played -> play
# This is called porterstemming

from nltk.stem import PorterStemmer

In [15]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)


data["review"] = data["review"].apply(stemming)

In [16]:
data.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


7. Encoding

In [17]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

data["sentiment"] = le.fit_transform(data["sentiment"])

In [18]:
y = data["sentiment"]
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

8. Vectorization

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(max_features = 5000)

X = tf.fit_transform(data["review"])
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057439 stored elements and shape (49582, 5000)>

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)

In [21]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [22]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [23]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [24]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True)

Building RNN

In [25]:
import torch.nn as nn
import torch.optim as optim

In [32]:
class RNN(nn.Module):
    def __init__ (self, input_size, hidden_size = 128, num_layers = 1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        #RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first = True)

        #fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x,h0)

        out = self.fc(out[:, -1, :])
        
        return out

In [33]:
input_size = X_train.shape[1]
model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

Training the RNN

In [37]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for xb, yb in train_loader:
        optimizer.zero_grad()
        
        xb = xb.unsqueeze(1)

        outputs = model(xb)

        outputs = torch.sigmoid(outputs.squeeze())

        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        

    print(f"Epoch = {epoch+1}/{epochs}, Loss = {loss.item()}")

Epoch = 1/10, Loss = 0.3682164251804352
Epoch = 2/10, Loss = 0.21252618730068207
Epoch = 3/10, Loss = 0.1651156097650528
Epoch = 4/10, Loss = 0.22828209400177002
Epoch = 5/10, Loss = 0.4708307981491089
Epoch = 6/10, Loss = 0.1472814679145813
Epoch = 7/10, Loss = 0.445526123046875
Epoch = 8/10, Loss = 0.1668485850095749
Epoch = 9/10, Loss = 0.24796341359615326
Epoch = 10/10, Loss = 0.2596818208694458


Evaluate

In [39]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    total_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze())>0.5).float()

        total_vals += yb.size(0)
        correct_vals +=(predicted == yb).sum().item()

    print(f"Accuracy = {correct_vals/total_vals * 100}")

Accuracy = 85.55006554401533
